TO DO: Turn AS off and on, will this still work?

Beacon filtering now works properly! However, there's a small lag now when running physcial commands, try to fix next...

In [1]:
import socket
import time
import xml.etree.ElementTree as ET

class GilsonSession:
    """
    A class to manage a session with a Gilson Ethernet-controlled device.
    Handles connection, command formatting, sending, and response parsing.
    """

    def __init__(self, ip, admin_port=50185):
        self.ip = ip
        self.admin_port = admin_port
        self.session_port = None
        self.session_socket = None
        self.sequence_number = 40
        self._connect()
        self.send_admin_command()

    def _connect(self):
        message = "<Gilson><GilsonConnect>Gilson Ethernet Utility</GilsonConnect></Gilson>"

        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(5)
            s.connect((self.ip, self.admin_port))
            s.sendall(message.encode('ascii'))
            response = s.recv(4096).decode('ascii')

        if "<ParameterName>ResponseValue</ParameterName>" in response:
            port = int(response.split("<ParameterValue>")[2].split("</ParameterValue>")[0])
            self.session_port = port

            self.session_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.session_socket.settimeout(2)
            self.session_socket.connect((self.ip, self.session_port))
        else:
            raise RuntimeError("Failed to retrieve assigned session port from Gilson.")

    def send_raw_command(self, xml_payload, expected_command=None):
        try:
            self.session_socket.sendall(xml_payload.encode('ascii'))
            buffer = b""
            start_time = time.time()
            while time.time() - start_time < 1.0:
                try:
                    self.session_socket.settimeout(0.2)
                    part = self.session_socket.recv(8192)
                    if part:
                        buffer += part
                    else:
                        break
                except socket.timeout:
                    continue

            responses = buffer.decode('ascii', errors='ignore')
            messages = responses.split("</Message>")
            for msg in messages:
                msg = msg.strip()
                if not msg:
                    continue
                msg += "</Message>" if not msg.endswith("</Message>") else ""
                if expected_command:
                    if f"<CommandName>{expected_command}</CommandName>" in msg and "<Response>" in msg:
                        return msg
                else:
                    if "<Response>" in msg:
                        return msg
            return None
        except Exception as e:
            print("Error sending raw command:", e)
            return None

    def _get_next_sequence_number(self):
        self.sequence_number += 1
        return self.sequence_number

    def make_command(self, command_name, device="GX-27x", device_id=35, sequence_number=None, command_type="Local", parameters=None):
        if sequence_number is None:
            sequence_number = self._get_next_sequence_number()

        param_xml = ""
        if parameters:
            param_xml = "<Parameters>"
            for key, value in parameters.items():
                param_xml += f"<Parameter><ParameterName>{key}</ParameterName><ParameterValue>{value}</ParameterValue></Parameter>"
            param_xml += "</Parameters>"

        xml = f"""
<Gilson>
  <CommandData>
    <Commands>
      <Command>
        <CommandType>{command_type}</CommandType>
        <CommandInfo>
          <InstrumentInfo>
            <DeviceName>{device}</DeviceName>
            <DeviceId>{device_id}</DeviceId>
          </InstrumentInfo>
          <SequenceNumber>{sequence_number}</SequenceNumber>
          <Synchronize>True</Synchronize>
          <Selected>False</Selected>
          <CommandXML>
            <CommandName>{command_name}</CommandName>
            {param_xml}
          </CommandXML>
        </CommandInfo>
      </Command>
    </Commands>
  </CommandData>
</Gilson>"""
        return xml.strip()

    def send_command(self, command_name, device="GX-27x", device_id=35, sequence_number=None, parameters=None):
        xml = self.make_command(command_name, device, device_id, sequence_number, command_type="Local", parameters=parameters)
        return self.send_raw_command(xml, expected_command=command_name)

    def send_immediate_command(self, command_name, device="GX-27x", device_id=35, sequence_number=None, parameters=None):
        xml = self.make_command(command_name, device, device_id, sequence_number, command_type="Immediate", parameters=parameters)
        return self.send_raw_command(xml, expected_command=command_name)

    def send_admin_command(self):
        xml = """
<Gilson>
  <CommandData>
    <Commands>
      <Command>
        <CommandType>Local</CommandType>
        <CommandInfo>
          <InstrumentInfo>
            <DeviceName>System</DeviceName>
            <DeviceId>-1</DeviceId>
          </InstrumentInfo>
          <SequenceNumber>0</SequenceNumber>
          <Synchronize>True</Synchronize>
          <Selected>False</Selected>
          <CommandXML>
            <CommandName>Admin</CommandName>
          </CommandXML>
        </CommandInfo>
      </Command>
    </Commands>
  </CommandData>
</Gilson>"""
        return self.send_raw_command(xml.strip(), expected_command="Admin")

    def move_x(self, position):
        parameters = {"X Position": position}
        return self.send_command("Move X", parameters=parameters)

    def move_y(self, position):
        parameters = {"Y Position": position}
        return self.send_command("Move Y", parameters=parameters)

    def move_z(self, position):
        parameters = {"Z Position": position}
        return self.send_command("Move Z", parameters=parameters)

    def move_xy(self, x_position, y_position):
        parameters = {
            "X Position": x_position,
            "Y Position": y_position
        }
        return self.send_command("Move XY", parameters=parameters)

    def close(self):
        if self.session_socket:
            self.session_socket.close()
            self.session_socket = None


In [2]:
g = GilsonSession("192.168.1.101")

In [4]:
g.send_command("Identify")

'<Message MessageId="2523" BlockCount="1" BlockId="0"><Gilson><Response><SequenceNumber>42</SequenceNumber><CommandName>Identify</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Identity</ParameterName><ParameterValue>GX 271 v2.3.0.1</ParameterValue></ReturnParameter></ReturnParameters></ParameterValue></Parameter></Parameters></Response></Gilson></Message>'

In [8]:
g.send_immediate_command("Get XY Position")

'<Message MessageId="2561" BlockCount="1" BlockId="0"><Gilson><Response><SequenceNumber>51</SequenceNumber><CommandName>Get XY Position</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>X Position</ParameterName><ParameterValue>3.8</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Y Position</ParameterName><ParameterValue>2.3</ParameterValue></ReturnParameter></ReturnParameters></ParameterValue></Parameter></Parameters></Response></Gilson></Message>'

In [6]:
g.send_immediate_command("Home")
g.move_x("100")
g.move_y("100")
g.send_immediate_command("Home")
g.move_xy("100","100")
g.send_immediate_command("Home")

'<Message MessageId="2556" BlockCount="1" BlockId="0"><Gilson><Response><SequenceNumber>49</SequenceNumber><CommandName>Home</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue>Success</ParameterValue></Parameter></Parameters></Response></Gilson></Message>'

In [7]:
#g.close()